# CV Attempt Model — Does Delivery Velocity Add Discrimination?

Pools all per-attempt data across every runner we have CV deliveries for and asks:
**does the CV-derived velocity metric (`avg_velocity_ftps`) add prediction power over
Statcast lead distances alone?**

Three feature blocks compared with leave-one-out CV AUC:
- **BASE** — `lead_at_firstmove_ft`, `gain_to_release_ft`, `lead_at_release_ft`
- **+DELIVERY** — BASE + `delivery_s`
- **+VELOCITY** — BASE + `delivery_s` + `avg_velocity_ftps`

The headline is the delta: `AUC(+VELOCITY) − AUC(BASE)`.

> ⚠️ **Small-sample caveat:** with only ~3–5 CS per runner, the classification AUC is
> high-variance. Treat it as a proof-of-harness. The **univariate separation table** and
> **run-value regression** are the more stable reads.

**Covered here:**
- Archived per-runner runs: Naylor 2025 · Naylor 2026 · Soto 2025 · Soto 2026
- Pooled multi-runner model (`attempt_model.py`)


## Setup

In [ ]:
import sys, csv, os
import numpy as np
import pandas as pd
from pathlib import Path

REPO = Path().resolve()
while not (REPO / "Figures").exists() and REPO.parent != REPO:
    REPO = REPO.parent

CV   = REPO / "Computer Vision"
ARCH = CV / "archive" / "Archived Runner Runs"
PIPE = CV / "CV Detection Pipeline"

if str(PIPE) not in sys.path:
    sys.path.insert(0, str(PIPE))

print("Repo root:", REPO)


## Archived Runner Run: Naylor 2025

### statcast_ref — Naylor 2025

In [ ]:
#!/usr/bin/env python3
"""
Josh Naylor 2025 Statcast base-stealing reference + head-to-head vs our CV metric.

IMPORTANT — Savant data sourcing (year correctness):
  * The basestealing-run-value LEADERBOARD CSV export IGNORES its ?year= param
    (it serves the current-season default regardless), so it CANNOT be used for a
    past-season reference. Verified: year=2025 and year=2026 return byte-identical
    rows (Naylor 9 SB / 1 CS both times).
  * The PER-ATTEMPT service
        /leaderboard/services/basestealing-running-game/{rid}?season_start=Y&season_end=Y
    DOES honor the year (2024->3, 2025->23, 2026->10 attempts). It is the authoritative
    source, and it is exactly what naylor2025_leads.csv was built from. So we compute
    the season SB-tracking anchors (SB/CS, run value, leads) from the leads CSV.
  * The sprint_speed leaderboard DOES honor ?min_season/?max_season, so sprint speed
    and home-to-1B are pulled live from there for the correct year.

Writes:
  statcast_ref_2025.csv         Naylor's season SB metrics (one row, year-correct)
  metric_vs_statcast_2025.csv   per-play velocity joined w/ season anchors

Network: Savant has no sandbox DNS -> run with dangerouslyDisableSandbox.
"""
import csv, io, os, statistics as st, requests

HERE = os.path.dirname(os.path.abspath(__file__))
NAYLOR = 647304
YEAR = 2025
S = requests.Session(); S.headers.update({"User-Agent": "Mozilla/5.0"})

SPRINT = ("https://baseballsavant.mlb.com/leaderboard/sprint_speed"
          f"?attempts=1&min_season={YEAR}&max_season={YEAR}&position=&team=&csv=true")


def getcsv(u):
    return list(csv.DictReader(io.StringIO(S.get(u, timeout=30).text.lstrip("﻿"))))


def fnum(x):
    try: return float(x)
    except (TypeError, ValueError): return None


def main():
    # ---- sprint speed (year-honored leaderboard) ----
    sprint = getcsv(SPRINT)
    s = next(r for r in sprint if r.get("player_id") == str(NAYLOR))

    # ---- season SB-tracking anchors computed from per-attempt leads (year-correct) ----
    leads = list(csv.DictReader(open(os.path.join(HERE, "naylor2025_leads.csv"))))
    sb_rows = [r for r in leads if r["result"] == "SB"]
    cs_rows = [r for r in leads if r["result"] == "CS"]
    n_sb, n_cs = len(sb_rows), len(cs_rows)
    fm_all = [fnum(r["lead_at_firstmove_ft"]) for r in leads]
    rel_all = [fnum(r["lead_at_release_ft"]) for r in leads]
    gain_all = [fnum(r["gain_to_release_ft"]) for r in leads]
    fm_sb = [fnum(r["lead_at_firstmove_ft"]) for r in sb_rows]
    rel_sb = [fnum(r["lead_at_release_ft"]) for r in sb_rows]

    ref = {
        "player": "Josh Naylor", "season": YEAR,
        "sprint_speed_ftps": s["sprint_speed"],
        "hp_to_1b_s": s["hp_to_1b"],
        "competitive_runs": s["competitive_runs"],
        "n_tracked_attempts": len(leads),
        "n_sb": n_sb, "n_cs": n_cs,
        "sb_success_pct": round(100 * n_sb / (n_sb + n_cs), 1),
        "steal_run_value": round(sum(fnum(r["run_value"]) for r in leads), 3),
        "primary_lead_ft": round(st.mean(fm_all), 2),
        "secondary_lead_ft": round(st.mean(rel_all), 2),
        "sec_minus_prim_lead_ft": round(st.mean(gain_all), 2),
        "primary_lead_sbx_ft": round(st.mean(fm_sb), 2),
        "secondary_lead_sbx_ft": round(st.mean(rel_sb), 2),
        "source_note": "SB anchors from per-attempt basestealing-running-game (year-correct); "
                       "sprint from sprint_speed leaderboard; leaderboard CSV year-param is broken",
    }
    with open(os.path.join(HERE, "statcast_ref_2025.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(ref)); w.writeheader(); w.writerow(ref)

    # ---- our per-play metric ----
    dv = list(csv.DictReader(open(os.path.join(HERE, "delivery_velocity_2025.csv"))))
    usable = [r for r in dv if r["avg_velocity_ftps"] != ""]
    vel = [float(r["avg_velocity_ftps"]) for r in usable]
    lfm = [float(r["lead_at_firstmove_ft"]) for r in dv]
    lrel = [float(r["lead_at_release_ft"]) for r in dv]
    sprint_v = fnum(ref["sprint_speed_ftps"])

    print("=" * 78)
    print(f"JOSH NAYLOR {YEAR} — SB-TRACKING METRICS  (Statcast season + our CV metric)")
    print("=" * 78)
    print("SEASON STATCAST (year-correct; SB anchors from per-attempt service):")
    print(f"  sprint speed          {ref['sprint_speed_ftps']:>6} ft/s")
    print(f"  home-to-1B            {ref['hp_to_1b_s']:>6} s")
    print(f"  tracked attempts      {ref['n_tracked_attempts']:>6}")
    print(f"  primary lead (all)    {ref['primary_lead_ft']:>6} ft")
    print(f"  secondary lead (all)  {ref['secondary_lead_ft']:>6} ft")
    print(f"  sec-minus-prim gain   {ref['sec_minus_prim_lead_ft']:>6} ft")
    print(f"  primary lead (steals) {ref['primary_lead_sbx_ft']:>6} ft")
    print(f"  secondary lead(steals){ref['secondary_lead_sbx_ft']:>6} ft")
    print(f"  SB / CS               {ref['n_sb']} / {ref['n_cs']}  "
          f"({ref['sb_success_pct']}% success)")
    print(f"  steal run value       {ref['steal_run_value']:>+6} runs")
    print()
    print("OUR CV METRIC (delivery-window velocity, per-attempt — varies by matchup):")
    print(f"  mean {st.mean(vel):.2f}  median {st.median(vel):.2f}  "
          f"range {min(vel):.2f}-{max(vel):.2f} ft/s   (n={len(vel)})")
    print()
    print("-" * 78)
    print("CROSS-VALIDATION  (these are the same per-attempt rows, sanity check):")
    print(f"  mean lead@firstmove = {st.mean(lfm):5.2f} ft   (= primary lead all = {ref['primary_lead_ft']})")
    print(f"  mean lead@release   = {st.mean(lrel):5.2f} ft   (= secondary lead all = {ref['secondary_lead_ft']})")
    print()
    print("-" * 78)
    print(f"HEAD-TO-HEAD — what each metric can/cannot do for these {len(dv)} attempts:")
    print(f"  sprint speed ({sprint_v:.1f} ft/s): runner-only, SAME every play -> 0 per-play")
    print(f"     resolution; aggregates can't tell a good matchup from a bad one.")
    print(f"  our velocity (mean {st.mean(vel):.1f} ft/s): runner x PITCHER interaction ->")
    print(f"     varies {min(vel):.1f}-{max(vel):.1f} ft/s across matchups; encodes the delivery he")
    print(f"     exploited. The lone CS (Littell) is the LOWEST velocity in the set.")
    print(f"  ratio our-vel / sprint-speed = {st.mean(vel)/sprint_v:.2f}  "
          f"(~{100*st.mean(vel)/sprint_v:.0f}% of top speed during the delivery window)")
    print("=" * 78)

    rows = [(float(r["avg_velocity_ftps"]) if r["avg_velocity_ftps"] != "" else -1, r) for r in dv]
    rows.sort(key=lambda x: -x[0])
    out = []
    for v, r in rows:
        out.append({
            "date": r["date"], "pitcher_name": r["pitcher_name"], "result": r["result"],
            "avg_velocity_ftps": r["avg_velocity_ftps"],
            "lead_at_release_ft": r["lead_at_release_ft"],
            "delivery_s": r["delivery_s"], "analysis_method": r["analysis_method"],
            "naylor_sprint_speed_ftps": ref["sprint_speed_ftps"],
            "naylor_secondary_lead_sbx_ft": ref["secondary_lead_sbx_ft"],
        })
    with open(os.path.join(HERE, "metric_vs_statcast_2025.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(out[0])); w.writeheader(); w.writerows(out)
    print("\nwrote statcast_ref_2025.csv + metric_vs_statcast_2025.csv")


if __name__ == "__main__":
    main()


### delivery_velocity — Naylor 2025

In [ ]:
#!/usr/bin/env python3
"""
Delivery-window runner velocity metric (Josh Naylor, 2025).

    avg_velocity_ftps = gain_to_release_ft / delivery_s

gain_to_release_ft = lead_at_release_ft - lead_at_firstmove_ft   (Statcast lead data)
delivery_s         = CV-measured first-move -> ball-release time  (cv_pilot detector)

The average ground speed (ft/s) the runner builds during the window the pitcher is
committed to the plate. Joins on play_id (the Savant GUID) so it is robust to
duplicate (date, pitcher) pairs (Naylor stole twice off Hendricks, Martin, Eisert
on the same day in 2025).

Inputs (same folder):
    naylor2025_leads.csv          Statcast lead distances + outcome/run value
    pitcher_delivery_2025.csv     CV per-clip delivery time + confidence (100% coverage)

Output:
    delivery_velocity_2025.csv    joined rows + avg_velocity_ftps
"""
import csv, os, statistics as st

HERE = os.path.dirname(os.path.abspath(__file__))
LEADS = os.path.join(HERE, "naylor2025_leads.csv")
DELIV = os.path.join(HERE, "pitcher_delivery_2025.csv")
OUT = os.path.join(HERE, "delivery_velocity_2025.csv")


def load_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def fnum(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def main():
    leads = load_csv(LEADS)
    deliv = {r["play_id"]: r for r in load_csv(DELIV)}

    rows = []
    for L in leads:
        d = deliv.get(L["play_id"])
        if d is None:
            continue
        delivery_s = fnum(d.get("delivery_s"))
        usable = str(d.get("usable", "")).strip().lower() == "true"
        gain = fnum(L.get("gain_to_release_ft"))
        avg_vel = None
        if usable and delivery_s and delivery_s > 0 and gain is not None:
            avg_vel = round(gain / delivery_s, 3)
        rows.append({
            "date": L["date"],
            "pitcher_name": L["pitcher_name"],
            "pitcher_id": L["pitcher_id"],
            "p_throws": d.get("p_throws", ""),
            "base": L["base"],
            "result": L["result"],
            "run_value": L["run_value"],
            "lead_at_firstmove_ft": L["lead_at_firstmove_ft"],
            "lead_at_release_ft": L["lead_at_release_ft"],
            "gain_to_release_ft": L["gain_to_release_ft"],
            "delivery_s": d.get("delivery_s", ""),
            "release_conf": d.get("release_conf", ""),
            "usable": d.get("usable", ""),
            "analysis_method": d.get("analysis_method", ""),
            "play_id": L["play_id"],
            "avg_velocity_ftps": "" if avg_vel is None else avg_vel,
        })

    cols = ["date", "pitcher_name", "pitcher_id", "p_throws", "base", "result",
            "run_value", "lead_at_firstmove_ft", "lead_at_release_ft",
            "gain_to_release_ft", "delivery_s", "release_conf", "usable",
            "analysis_method", "play_id", "avg_velocity_ftps"]
    with open(OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader(); w.writerows(rows)

    usable_rows = [r for r in rows if r["avg_velocity_ftps"] != ""]
    vals = [float(r["avg_velocity_ftps"]) for r in usable_rows]

    print("=" * 74)
    print("DELIVERY-WINDOW RUNNER VELOCITY  (Josh Naylor, 2025)")
    print("  avg_velocity_ftps = gain_to_release_ft / delivery_s")
    print("=" * 74)
    hdr = f"{'date':10} {'pitcher':22} {'res':4} {'gain':>5} {'deliv':>6} {'vel':>7} {'meth':>9}"
    print(hdr); print("-" * len(hdr))
    for r in sorted(rows, key=lambda x: (x["avg_velocity_ftps"] == "",
                    -(float(x["avg_velocity_ftps"]) if x["avg_velocity_ftps"] != "" else 0))):
        vel = r["avg_velocity_ftps"]
        vel_s = f"{float(vel):7.2f}" if vel != "" else "    n/a"
        dv = r["delivery_s"] or "n/a"
        print(f"{r['date']:10} {r['pitcher_name']:22} {r['result']:4} "
              f"{float(r['gain_to_release_ft']):5.1f} {str(dv):>6} {vel_s} "
              f"{r['analysis_method']:>9}")
    print("-" * len(hdr))
    print(f"usable plays: {len(usable_rows)}/{len(rows)}")
    if vals:
        print(f"velocity ft/s  mean={st.mean(vals):.2f}  median={st.median(vals):.2f}  "
              f"min={min(vals):.2f}  max={max(vals):.2f}  std={st.pstdev(vals):.2f}")

    sb = [float(r["avg_velocity_ftps"]) for r in usable_rows if r["result"] == "SB"]
    cs = [float(r["avg_velocity_ftps"]) for r in usable_rows if r["result"] == "CS"]
    print()
    if sb:
        print(f"  SB (n={len(sb)})  mean vel = {st.mean(sb):.2f} ft/s")
    if cs:
        print(f"  CS (n={len(cs)})  mean vel = {st.mean(cs):.2f} ft/s  "
              f"<- single sample (Littell); illustrative only")

    def corr(xs, ys):
        if len(xs) < 3:
            return None
        mx, my = st.mean(xs), st.mean(ys)
        num = sum((a - mx) * (b - my) for a, b in zip(xs, ys))
        dx = sum((a - mx) ** 2 for a in xs) ** 0.5
        dy = sum((b - my) ** 2 for b in ys) ** 0.5
        return num / (dx * dy) if dx and dy else None

    dlv = [float(r["delivery_s"]) for r in usable_rows]
    gns = [float(r["gain_to_release_ft"]) for r in usable_rows]
    lfm = [float(r["lead_at_firstmove_ft"]) for r in usable_rows]
    print()
    for label, c in (("delivery_s", corr(vals, dlv)),
                     ("gain_to_release_ft", corr(vals, gns)),
                     ("lead_at_firstmove", corr(vals, lfm))):
        if c is not None:
            print(f"  corr(velocity, {label:20}) = {c:+.2f}")
    print("=" * 74)
    print(f"wrote {OUT}")


if __name__ == "__main__":
    main()


### build_delivery — Naylor 2025

In [ ]:
#!/usr/bin/env python3
"""
Assemble pitcher_delivery_2025.csv (mirrors the 2026 schema) from:
  - pilot_results.csv         full detector pass over all 23 clips
  - /tmp/rec_<clip>.csv        per-clip 6s-cap recovery runs (broadcast-follows-runner)
  - naylor2025_leads.csv       date / base / result / run_value join keys
  - reuse-fallback             a same-pitcher measured delivery when a 2025 clip is
                               un-measurable even after the cap

Selection per clip:
  1. if the full pass produced usable=True -> use it (analysis_method=full)
  2. else if the 6s-cap recovery produced usable=True -> use it (analysis_method=capped_6s)
  3. else apply reuse-fallback (analysis_method=reused, note explains the source)

The reuse table is intentionally explicit (small, auditable) rather than auto-joined.
"""
import csv, os

HERE = os.path.dirname(os.path.abspath(__file__))
PILOT = os.path.join(HERE, "pilot_results.csv")
LEADS = os.path.join(HERE, "naylor2025_leads.csv")
OUT = os.path.join(HERE, "pitcher_delivery_2025.csv")

# clips re-run with the 6s analysis cap (whether or not they recovered)
RECOVERED = [
    "NAYLOR_03ee0445_kerkering_R", "NAYLOR_04855f81_gausman_R",
    "NAYLOR_09e58c6b_lopez_R", "NAYLOR_0a8f1dbd_eisert_L",
    "NAYLOR_39a80ca9_martin_R", "NAYLOR_8f12e2af_hendricks_R",
    "NAYLOR_af7bfe85_littell_R",
]

# reuse-fallback: play_id -> (delivery_s, note)  for clips un-measurable even after cap
REUSE = {
    # Gausman 2025 clip un-measurable (camera follow); reuse 20-pitcher SB pool median
    "04855f81-6b86-3b37-a53a-c5df4362065d": (1.0932, "reused: Gausman pool median (cv_pilot/pitcher_delivery_sb.csv)"),
    # Hendricks 8f12e2af un-measurable; reuse his OTHER 2025 clip (718944e7) measured 0.9322
    "8f12e2af-0a21-3542-8ce5-1bf7860cf0a3": (0.9322, "reused: Hendricks 2025 same-pitcher clip 718944e7"),
}


def load(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def truthy(x):
    return str(x).strip().lower() == "true"


def main():
    pilot = {r["play_id"]: r for r in load(PILOT)}
    leads = {r["play_id"]: r for r in load(LEADS)}

    rec = {}
    for c in RECOVERED:
        p = f"/tmp/rec_{c}.csv"
        if os.path.exists(p):
            row = load(p)[0]
            rec[row["play_id"]] = row

    out_rows = []
    for play_id, L in leads.items():
        base = L["base"]; result = L["result"]
        event = ("Caught Stealing " if result == "CS" else "Stolen Base ") + base
        p = pilot.get(play_id, {})
        r = rec.get(play_id)

        delivery_s = usable = release_conf = method = note = None

        if p and truthy(p.get("usable")):
            delivery_s = p.get("delivery_s"); usable = True
            release_conf = p.get("release_kpt_conf"); method = "full"
        elif r and truthy(r.get("usable")):
            delivery_s = r.get("delivery_s"); usable = True
            release_conf = r.get("release_kpt_conf"); method = "capped_6s"
            note = "recovered via 6s analysis cap (broadcast follows runner)"
        elif play_id in REUSE:
            delivery_s, note = REUSE[play_id]
            usable = True; release_conf = ""; method = "reused"
        else:
            # no measurement and no fallback -> keep the (failed) full-pass number, mark unusable
            delivery_s = p.get("delivery_s", ""); usable = False
            release_conf = p.get("release_kpt_conf", ""); method = "full"
            note = "un-measurable; no same-pitcher fallback available"

        out_rows.append({
            "date": L["date"],
            "pitcher_id": L["pitcher_id"],
            "pitcher_name": L["pitcher_name"],
            "p_throws": p.get("p_throws", r.get("p_throws", "") if r else ""),
            "event": event,
            "catcher_name": L["catcher_name"],
            "clip_id": p.get("clip_id", r.get("clip_id", "") if r else ""),
            "play_id": play_id,
            "fps": p.get("fps", ""),
            "delivery_s": delivery_s,
            "usable": usable,
            "release_conf": release_conf,
            "analysis_method": method,
            "note": note or "",
        })

    out_rows.sort(key=lambda x: x["date"])
    cols = ["date", "pitcher_id", "pitcher_name", "p_throws", "event",
            "catcher_name", "clip_id", "play_id", "fps", "delivery_s",
            "usable", "release_conf", "analysis_method", "note"]
    with open(OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader(); w.writerows(out_rows)

    n_us = sum(1 for r in out_rows if r["usable"])
    n_re = sum(1 for r in out_rows if r["analysis_method"] == "reused")
    n_cap = sum(1 for r in out_rows if r["analysis_method"] == "capped_6s")
    print(f"[write] {OUT}  ({len(out_rows)} attempts; {n_us} usable; "
          f"{n_cap} capped_6s; {n_re} reused)")
    for r in out_rows:
        if not r["usable"] or r["analysis_method"] != "full":
            print(f"  {r['date']} {r['pitcher_name']:22} {r['analysis_method']:9} "
                  f"d={r['delivery_s']} usable={r['usable']}  {r['note']}")


if __name__ == "__main__":
    main()


### Load Naylor 2025 results

In [ ]:
naylor25 = ARCH / "Naylor_2025" / "delivery_velocity_2025.csv"
if naylor25.exists():
    df = pd.read_csv(naylor25)
    print(f"Naylor 2025: {len(df)} attempts")
    display(df[["date","pitcher_name","result","lead_at_firstmove_ft",
                "gain_to_release_ft","delivery_s","avg_velocity_ftps"]].head(10))
else:
    print(f"Not found: {naylor25}  (run the CV Detection Pipeline first)")


## Archived Runner Run: Naylor 2026

### statcast_ref — Naylor 2026

In [ ]:
#!/usr/bin/env python3
"""
Pull Josh Naylor's 2026 Statcast base-stealing reference metrics from Baseball
Savant and rank our delivery-window velocity metric head-to-head against them.

Savant hosts (note: NO hyphen):
  - basestealing-run-value leaderboard (runner view) -> leads, run value, SB/CS
  - sprint_speed leaderboard                          -> sprint speed, hp_to_1b

Writes:
  statcast_ref_2026.csv         Naylor's season SB metrics (one row)
  metric_vs_statcast_2026.csv   per-play velocity joined w/ season anchors
Prints the head-to-head read-out.
"""
import csv, io, os, statistics as st, requests

HERE = os.path.dirname(os.path.abspath(__file__))
NAYLOR = 647304
UA = {"User-Agent": "Mozilla/5.0"}
S = requests.Session(); S.headers.update(UA)

BSRV = ("https://baseballsavant.mlb.com/leaderboard/basestealing-run-value"
        "?type=runner&year=2026&team=&min=0&sortColumn=runner_runs_swiped_total"
        "&sortDirection=desc&csv=true")
SPRINT = ("https://baseballsavant.mlb.com/leaderboard/sprint_speed"
          "?attempts=1&min_season=2026&max_season=2026&position=&team=&csv=true")


def getcsv(u):
    t = S.get(u, timeout=30).text.lstrip("﻿")
    return list(csv.DictReader(io.StringIO(t)))


def fnum(x):
    try: return float(x)
    except (TypeError, ValueError): return None


def main():
    bsrv = getcsv(BSRV)
    sprint = getcsv(SPRINT)
    b = next(r for r in bsrv if r.get("player_id") == str(NAYLOR))
    s = next(r for r in sprint if r.get("player_id") == str(NAYLOR))

    ref = {
        "player": "Josh Naylor", "season": 2026,
        "sprint_speed_ftps": s["sprint_speed"],
        "hp_to_1b_s": s["hp_to_1b"],
        "competitive_runs": s["competitive_runs"],
        "n_sb": b["n_sb"], "n_cs": b["n_cs"],
        "sb_success_pct": round(100 * int(b["n_sb"]) / (int(b["n_sb"]) + int(b["n_cs"])), 1),
        "steal_run_value": round(fnum(b["runs_stolen_on_running_act"]), 3),
        "attempt_rate_sbx": b["rate_sbx"],
        "n_init": b["n_init"],
        "primary_lead_ft": round(fnum(b["r_primary_lead"]), 2),
        "secondary_lead_ft": round(fnum(b["r_secondary_lead"]), 2),
        "sec_minus_prim_lead_ft": round(fnum(b["r_sec_minus_prim_lead"]), 2),
        "primary_lead_sbx_ft": round(fnum(b["r_primary_lead_sbx"]), 2),
        "secondary_lead_sbx_ft": round(fnum(b["r_secondary_lead_sbx"]), 2),
    }
    with open(os.path.join(HERE, "statcast_ref_2026.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(ref)); w.writeheader(); w.writerow(ref)

    # ---- our per-play metric ----
    dv = list(csv.DictReader(open(os.path.join(HERE, "delivery_velocity_2026.csv"))))
    usable = [r for r in dv if r["avg_velocity_ftps"] != ""]
    vel = [float(r["avg_velocity_ftps"]) for r in usable]
    lfm = [float(r["lead_at_firstmove_ft"]) for r in dv]
    lrel = [float(r["lead_at_release_ft"]) for r in dv]

    sprint_v = fnum(ref["sprint_speed_ftps"])

    print("=" * 78)
    print("JOSH NAYLOR 2026 — SB-TRACKING METRICS  (Statcast season + our CV metric)")
    print("=" * 78)
    print("SEASON STATCAST (runner-only, constant across all attempts):")
    print(f"  sprint speed          {ref['sprint_speed_ftps']:>6} ft/s   "
          f"(MLB avg ~27.0 -> Naylor is SLOW, ~12th pctile)")
    print(f"  home-to-1B            {ref['hp_to_1b_s']:>6} s")
    print(f"  primary lead          {ref['primary_lead_ft']:>6} ft")
    print(f"  secondary lead        {ref['secondary_lead_ft']:>6} ft")
    print(f"  sec-minus-prim gain   {ref['sec_minus_prim_lead_ft']:>6} ft")
    print(f"  primary lead (steals) {ref['primary_lead_sbx_ft']:>6} ft")
    print(f"  secondary lead(steals){ref['secondary_lead_sbx_ft']:>6} ft")
    print(f"  SB / CS               {ref['n_sb']} / {ref['n_cs']}  "
          f"({ref['sb_success_pct']}% success)")
    print(f"  steal run value       {ref['steal_run_value']:>+6} runs")
    print(f"  attempt rate          {ref['attempt_rate_sbx']}  (of {ref['n_init']} opportunities)")
    print()
    print("OUR CV METRIC (delivery-window velocity, per-attempt — varies by matchup):")
    print(f"  mean {st.mean(vel):.2f}  median {st.median(vel):.2f}  "
          f"range {min(vel):.2f}-{max(vel):.2f} ft/s   (n={len(vel)})")
    print()
    print("-" * 78)
    print("CROSS-VALIDATION  (our per-play lead data vs Savant season anchors):")
    print(f"  our mean lead@firstmove = {st.mean(lfm):5.2f} ft   "
          f"Savant primary-lead(steals) = {ref['primary_lead_sbx_ft']} ft")
    print(f"  our mean lead@release   = {st.mean(lrel):5.2f} ft   "
          f"Savant secondary-lead(steals) = {ref['secondary_lead_sbx_ft']} ft  <- matches")
    print()
    print("-" * 78)
    print("HEAD-TO-HEAD — what each metric can and cannot do for these 10 attempts:")
    print(f"  sprint speed (24.5 ft/s): runner-only, SAME every play -> 0 per-play")
    print(f"     resolution; would predict Naylor a BAD base-stealer, yet he's 90% / +1.19 runs.")
    print(f"  our velocity (mean {st.mean(vel):.1f} ft/s): runner x PITCHER interaction ->")
    print(f"     varies {min(vel):.1f}-{max(vel):.1f} ft/s across matchups; encodes the delivery he")
    print(f"     exploited. This is the resolution sprint speed/lead aggregates lack.")
    print(f"  ratio our-vel / sprint-speed = {st.mean(vel)/sprint_v:.2f}  "
          f"(he reaches ~{100*st.mean(vel)/sprint_v:.0f}% of top speed during the delivery window)")
    print("=" * 78)

    # per-play table sorted by velocity, flag the CS
    rows = []
    for r in dv:
        v = r["avg_velocity_ftps"]
        rows.append((float(v) if v != "" else -1, r))
    rows.sort(key=lambda x: -x[0])
    print(f"{'pitcher':20} {'res':4} {'vel':>6} {'lead@rel':>9} {'deliv':>6}")
    out = []
    for v, r in rows:
        vs = f"{v:6.2f}" if v >= 0 else "   n/a"
        print(f"{r['pitcher_name']:20} {r['result']:4} {vs} "
              f"{float(r['lead_at_release_ft']):9.1f} {r['delivery_s'] or 'n/a':>6}")
        out.append({
            "pitcher_name": r["pitcher_name"], "result": r["result"],
            "avg_velocity_ftps": r["avg_velocity_ftps"],
            "lead_at_release_ft": r["lead_at_release_ft"],
            "delivery_s": r["delivery_s"],
            "naylor_sprint_speed_ftps": ref["sprint_speed_ftps"],
            "naylor_secondary_lead_sbx_ft": ref["secondary_lead_sbx_ft"],
        })
    with open(os.path.join(HERE, "metric_vs_statcast_2026.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(out[0])); w.writeheader(); w.writerows(out)
    print("\nwrote statcast_ref_2026.csv + metric_vs_statcast_2026.csv")


if __name__ == "__main__":
    main()


### delivery_velocity — Naylor 2026

In [ ]:
#!/usr/bin/env python3
"""
Delivery-window runner velocity metric (Josh Naylor, 2026).

Metric:
    avg_velocity_ftps = gain_to_release_ft / delivery_s

where
    gain_to_release_ft = lead_at_release_ft - lead_at_firstmove_ft   (Statcast lead data)
    delivery_s         = CV-measured first-move -> ball-release time  (cv_pilot detector)

This is the average ground speed (ft/s) the runner builds during the window the
pitcher is committed to the plate (first move -> release). It blends the runner's
secondary-lead burst with how much time the pitcher's delivery gives him. A high
value means the runner is covering a lot of ground in the exact window where the
battery can no longer hold him -- the cleanest single-number "free distance" proxy
we can build from the clips we have.

Inputs (same folder):
    naylor2026_leads.csv          Statcast lead distances + outcome/run value
    pitcher_delivery_2026.csv     CV per-clip delivery time + confidence

Output:
    delivery_velocity_2026.csv    joined rows + avg_velocity_ftps
Prints a performance read-out to stdout.
"""
import csv
import os
import statistics as st

HERE = os.path.dirname(os.path.abspath(__file__))
LEADS = os.path.join(HERE, "naylor2026_leads.csv")
DELIV = os.path.join(HERE, "pitcher_delivery_2026.csv")
OUT = os.path.join(HERE, "delivery_velocity_2026.csv")


def load_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def fnum(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def main():
    leads = load_csv(LEADS)
    deliv = load_csv(DELIV)

    # key on (date, pitcher_name): unique per play in this sample
    dkey = {(r["date"], r["pitcher_name"]): r for r in deliv}

    rows = []
    for L in leads:
        d = dkey.get((L["date"], L["pitcher_name"]))
        if d is None:
            continue
        delivery_s = fnum(d.get("delivery_s"))
        usable = (d.get("usable", "").strip().lower() == "true")
        gain = fnum(L.get("gain_to_release_ft"))
        avg_vel = None
        if usable and delivery_s and delivery_s > 0 and gain is not None:
            avg_vel = round(gain / delivery_s, 3)
        rows.append({
            "date": L["date"],
            "pitcher_name": L["pitcher_name"],
            "p_throws": d.get("p_throws", ""),
            "base": L["base"],
            "result": L["result"],
            "run_value": L["run_value"],
            "lead_at_firstmove_ft": L["lead_at_firstmove_ft"],
            "lead_at_release_ft": L["lead_at_release_ft"],
            "gain_to_release_ft": L["gain_to_release_ft"],
            "delivery_s": d.get("delivery_s", ""),
            "release_conf": d.get("release_conf", ""),
            "usable": d.get("usable", ""),
            "analysis_method": d.get("analysis_method", ""),
            "avg_velocity_ftps": "" if avg_vel is None else avg_vel,
        })

    cols = ["date", "pitcher_name", "p_throws", "base", "result", "run_value",
            "lead_at_firstmove_ft", "lead_at_release_ft", "gain_to_release_ft",
            "delivery_s", "release_conf", "usable", "analysis_method",
            "avg_velocity_ftps"]
    with open(OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        w.writerows(rows)

    # ---------- performance read-out ----------
    usable_rows = [r for r in rows if r["avg_velocity_ftps"] != ""]
    vals = [float(r["avg_velocity_ftps"]) for r in usable_rows]

    print("=" * 74)
    print("DELIVERY-WINDOW RUNNER VELOCITY  (Josh Naylor, 2026)")
    print("  avg_velocity_ftps = gain_to_release_ft / delivery_s")
    print("=" * 74)
    hdr = f"{'date':10} {'pitcher':20} {'res':4} {'gain':>5} {'deliv':>6} {'vel':>7} {'conf':>5}"
    print(hdr)
    print("-" * len(hdr))
    for r in sorted(rows, key=lambda x: (x["avg_velocity_ftps"] == "",
                                         -(float(x["avg_velocity_ftps"]) if x["avg_velocity_ftps"] != "" else 0))):
        vel = r["avg_velocity_ftps"]
        vel_s = f"{float(vel):7.2f}" if vel != "" else "    n/a"
        dv = r["delivery_s"] or "n/a"
        print(f"{r['date']:10} {r['pitcher_name']:20} {r['result']:4} "
              f"{float(r['gain_to_release_ft']):5.1f} {str(dv):>6} {vel_s} "
              f"{r['release_conf']:>5}")

    print("-" * len(hdr))
    print(f"usable plays: {len(usable_rows)}/{len(rows)}  "
          f"(excluded: {[r['pitcher_name'] for r in rows if r['avg_velocity_ftps']=='']})")
    if vals:
        print(f"velocity ft/s  mean={st.mean(vals):.2f}  median={st.median(vals):.2f}  "
              f"min={min(vals):.2f}  max={max(vals):.2f}  "
              f"std={st.pstdev(vals):.2f}")

    # SB vs CS split (only 1 CS in sample -> illustrative)
    sb = [float(r["avg_velocity_ftps"]) for r in usable_rows if r["result"] == "SB"]
    cs = [float(r["avg_velocity_ftps"]) for r in usable_rows if r["result"] == "CS"]
    print()
    if sb:
        print(f"  SB (n={len(sb)})  mean vel = {st.mean(sb):.2f} ft/s")
    if cs:
        print(f"  CS (n={len(cs)})  mean vel = {st.mean(cs):.2f} ft/s   "
              f"<- single sample (Joe Ryan); illustrative only")

    # correlation of velocity with delivery_s and with lead_at_firstmove
    def corr(xs, ys):
        if len(xs) < 3:
            return None
        mx, my = st.mean(xs), st.mean(ys)
        num = sum((a - mx) * (b - my) for a, b in zip(xs, ys))
        dx = sum((a - mx) ** 2 for a in xs) ** 0.5
        dy = sum((b - my) ** 2 for b in ys) ** 0.5
        return num / (dx * dy) if dx and dy else None

    dlv = [float(r["delivery_s"]) for r in usable_rows]
    gns = [float(r["gain_to_release_ft"]) for r in usable_rows]
    lfm = [float(r["lead_at_firstmove_ft"]) for r in usable_rows]
    print()
    c1 = corr(vals, dlv)
    c2 = corr(vals, gns)
    c3 = corr(vals, lfm)
    if c1 is not None:
        print(f"  corr(velocity, delivery_s)          = {c1:+.2f}  "
              f"(neg = slower deliveries give LOWER ft/s for same gain)")
    if c2 is not None:
        print(f"  corr(velocity, gain_to_release_ft)  = {c2:+.2f}  "
              f"(pos = more ground gained drives the metric)")
    if c3 is not None:
        print(f"  corr(velocity, lead_at_firstmove)   = {c3:+.2f}")
    print("=" * 74)
    print(f"wrote {OUT}")


if __name__ == "__main__":
    main()


In [ ]:
naylor26 = ARCH / "Naylor_2026" / "delivery_velocity_2026.csv"
if naylor26.exists():
    df = pd.read_csv(naylor26)
    print(f"Naylor 2026: {len(df)} attempts")
    display(df[["date","pitcher_name","result","gain_to_release_ft",
                "delivery_s","avg_velocity_ftps"]].head(10))
else:
    print(f"Not found: {naylor26}")


## Archived Runner Run: Soto 2025

### statcast_ref — Soto 2025

In [ ]:
#!/usr/bin/env python3
"""
Juan Soto 2025 Statcast base-stealing reference + head-to-head vs our CV metric.

IMPORTANT — Savant data sourcing (year correctness):
  * The basestealing-run-value LEADERBOARD CSV export IGNORES its ?year= param, so it
    CANNOT be used for a past-season reference. The PER-ATTEMPT service
        /leaderboard/services/basestealing-running-game/{rid}?season_start=Y&season_end=Y
    DOES honor the year and is what soto2025_leads.csv was built from, so the season
    SB-tracking anchors are computed from the leads CSV.
  * The sprint_speed leaderboard DOES honor ?min_season/?max_season.

Soto is a FAST runner (elite sprint speed) — the opposite profile to Naylor — so the
head-to-head asks whether his delivery-window velocity simply mirrors his top speed or
still varies by pitcher matchup.

Writes:
  statcast_ref_2025.csv         Soto's season SB metrics (one row, year-correct)
  metric_vs_statcast_2025.csv   per-play velocity joined w/ season anchors

Network: Savant has no sandbox DNS -> run with dangerouslyDisableSandbox.
"""
import csv, io, os, statistics as st, requests

HERE = os.path.dirname(os.path.abspath(__file__))
RUNNER_ID = 665742
RUNNER_NAME = "Juan Soto"
YEAR = 2025
LEADS = os.path.join(HERE, "soto2025_leads.csv")
DV = os.path.join(HERE, "delivery_velocity_2025.csv")
REF_OUT = os.path.join(HERE, "statcast_ref_2025.csv")
H2H_OUT = os.path.join(HERE, "metric_vs_statcast_2025.csv")
S = requests.Session(); S.headers.update({"User-Agent": "Mozilla/5.0"})

SPRINT = ("https://baseballsavant.mlb.com/leaderboard/sprint_speed"
          f"?attempts=1&min_season={YEAR}&max_season={YEAR}&position=&team=&csv=true")


def getcsv(u):
    return list(csv.DictReader(io.StringIO(S.get(u, timeout=30).text.lstrip("﻿"))))


def fnum(x):
    try: return float(x)
    except (TypeError, ValueError): return None


def main():
    sprint = getcsv(SPRINT)
    s = next(r for r in sprint if r.get("player_id") == str(RUNNER_ID))

    leads = list(csv.DictReader(open(LEADS)))
    sb_rows = [r for r in leads if r["result"] == "SB"]
    cs_rows = [r for r in leads if r["result"] == "CS"]
    n_sb, n_cs = len(sb_rows), len(cs_rows)
    # only rows with measured leads (some balk/FB rows have empty lead fields)
    lead_rows = [r for r in leads if fnum(r["lead_at_firstmove_ft"]) is not None]
    fm_all = [fnum(r["lead_at_firstmove_ft"]) for r in lead_rows]
    rel_all = [fnum(r["lead_at_release_ft"]) for r in lead_rows]
    gain_all = [fnum(r["gain_to_release_ft"]) for r in lead_rows]
    fm_sb = [fnum(r["lead_at_firstmove_ft"]) for r in sb_rows if fnum(r["lead_at_firstmove_ft"]) is not None]
    rel_sb = [fnum(r["lead_at_release_ft"]) for r in sb_rows if fnum(r["lead_at_release_ft"]) is not None]

    ref = {
        "player": RUNNER_NAME, "season": YEAR,
        "sprint_speed_ftps": s["sprint_speed"],
        "hp_to_1b_s": s["hp_to_1b"],
        "competitive_runs": s["competitive_runs"],
        "n_tracked_attempts": len(leads),
        "n_sb": n_sb, "n_cs": n_cs,
        "sb_success_pct": round(100 * n_sb / (n_sb + n_cs), 1) if (n_sb + n_cs) else "",
        "steal_run_value": round(sum(fnum(r["run_value"]) for r in leads if fnum(r["run_value"]) is not None), 3),
        "primary_lead_ft": round(st.mean(fm_all), 2),
        "secondary_lead_ft": round(st.mean(rel_all), 2),
        "sec_minus_prim_lead_ft": round(st.mean(gain_all), 2),
        "primary_lead_sbx_ft": round(st.mean(fm_sb), 2) if fm_sb else "",
        "secondary_lead_sbx_ft": round(st.mean(rel_sb), 2) if rel_sb else "",
        "source_note": "SB anchors from per-attempt basestealing-running-game (year-correct); "
                       "sprint from sprint_speed leaderboard; leaderboard CSV year-param is broken",
    }
    with open(REF_OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(ref)); w.writeheader(); w.writerow(ref)

    dv = list(csv.DictReader(open(DV)))
    usable = [r for r in dv if r["avg_velocity_ftps"] != ""]
    vel = [float(r["avg_velocity_ftps"]) for r in usable]
    lfm = [float(r["lead_at_firstmove_ft"]) for r in dv if r["lead_at_firstmove_ft"]]
    lrel = [float(r["lead_at_release_ft"]) for r in dv if r["lead_at_release_ft"]]
    sprint_v = fnum(ref["sprint_speed_ftps"])

    print("=" * 78)
    print(f"{RUNNER_NAME.upper()} {YEAR} — SB-TRACKING METRICS  (Statcast season + our CV metric)")
    print("=" * 78)
    print("SEASON STATCAST (year-correct; SB anchors from per-attempt service):")
    print(f"  sprint speed          {ref['sprint_speed_ftps']:>6} ft/s")
    print(f"  home-to-1B            {ref['hp_to_1b_s']:>6} s")
    print(f"  tracked attempts      {ref['n_tracked_attempts']:>6}")
    print(f"  primary lead (all)    {ref['primary_lead_ft']:>6} ft")
    print(f"  secondary lead (all)  {ref['secondary_lead_ft']:>6} ft")
    print(f"  sec-minus-prim gain   {ref['sec_minus_prim_lead_ft']:>6} ft")
    print(f"  primary lead (steals) {ref['primary_lead_sbx_ft']:>6} ft")
    print(f"  secondary lead(steals){ref['secondary_lead_sbx_ft']:>6} ft")
    print(f"  SB / CS               {ref['n_sb']} / {ref['n_cs']}  "
          f"({ref['sb_success_pct']}% success)")
    print(f"  steal run value       {ref['steal_run_value']:>+6} runs")
    print()
    if vel:
        print("OUR CV METRIC (delivery-window velocity, per-attempt — varies by matchup):")
        print(f"  mean {st.mean(vel):.2f}  median {st.median(vel):.2f}  "
              f"range {min(vel):.2f}-{max(vel):.2f} ft/s   (n={len(vel)})")
        print()
        print("-" * 78)
        print("CROSS-VALIDATION  (same per-attempt rows, sanity check):")
        if lfm:
            print(f"  mean lead@firstmove = {st.mean(lfm):5.2f} ft   (= primary lead all = {ref['primary_lead_ft']})")
        if lrel:
            print(f"  mean lead@release   = {st.mean(lrel):5.2f} ft   (= secondary lead all = {ref['secondary_lead_ft']})")
        print()
        print("-" * 78)
        print(f"HEAD-TO-HEAD — what each metric can/cannot do for these {len(usable)} attempts:")
        print(f"  sprint speed ({sprint_v:.1f} ft/s): runner-only, SAME every play -> 0 per-play resolution.")
        print(f"  our velocity (mean {st.mean(vel):.1f} ft/s): runner x PITCHER interaction ->")
        print(f"     varies {min(vel):.1f}-{max(vel):.1f} ft/s across matchups.")
        print(f"  ratio our-vel / sprint-speed = {st.mean(vel)/sprint_v:.2f}  "
              f"(~{100*st.mean(vel)/sprint_v:.0f}% of top speed during the delivery window)")
    print("=" * 78)

    rows = [(float(r["avg_velocity_ftps"]) if r["avg_velocity_ftps"] != "" else -1, r) for r in dv]
    rows.sort(key=lambda x: -x[0])
    out = []
    for v, r in rows:
        out.append({
            "date": r["date"], "pitcher_name": r["pitcher_name"], "result": r["result"],
            "avg_velocity_ftps": r["avg_velocity_ftps"],
            "lead_at_release_ft": r["lead_at_release_ft"],
            "delivery_s": r["delivery_s"], "analysis_method": r["analysis_method"],
            "soto_sprint_speed_ftps": ref["sprint_speed_ftps"],
            "soto_secondary_lead_sbx_ft": ref["secondary_lead_sbx_ft"],
        })
    with open(H2H_OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(out[0])); w.writeheader(); w.writerows(out)
    print(f"\nwrote {os.path.basename(REF_OUT)} + {os.path.basename(H2H_OUT)}")


if __name__ == "__main__":
    main()


### delivery_velocity — Soto 2025

In [ ]:
#!/usr/bin/env python3
"""
Delivery-window runner velocity metric (Juan Soto, 2025).

    avg_velocity_ftps = gain_to_release_ft / delivery_s

gain_to_release_ft = lead_at_release_ft - lead_at_firstmove_ft   (Statcast lead data)
delivery_s         = CV-measured first-move -> ball-release time  (cv_pilot detector)

The average ground speed (ft/s) the runner builds during the window the pitcher is
committed to the plate. Joins on play_id (the Savant GUID). Soto is a FAST runner
(contrast to Naylor) so this set probes whether a fast runner's delivery-window
velocity tracks his elite sprint speed or is still pitcher-matchup driven.

Inputs (same folder):
    soto2025_leads.csv            Statcast lead distances + outcome/run value
    pitcher_delivery_2025.csv     CV per-clip delivery time + confidence

Output:
    delivery_velocity_2025.csv    joined rows + avg_velocity_ftps
"""
import csv, os, statistics as st

HERE = os.path.dirname(os.path.abspath(__file__))
LEADS = os.path.join(HERE, "soto2025_leads.csv")
DELIV = os.path.join(HERE, "pitcher_delivery_2025.csv")
OUT = os.path.join(HERE, "delivery_velocity_2025.csv")
RUNNER = "Juan Soto"
YEAR = 2025


def load_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def fnum(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def main():
    leads = load_csv(LEADS)
    deliv = {r["play_id"]: r for r in load_csv(DELIV)}

    rows = []
    for L in leads:
        d = deliv.get(L["play_id"])
        if d is None:
            continue
        delivery_s = fnum(d.get("delivery_s"))
        usable = str(d.get("usable", "")).strip().lower() == "true"
        gain = fnum(L.get("gain_to_release_ft"))
        avg_vel = None
        if usable and delivery_s and delivery_s > 0 and gain is not None:
            avg_vel = round(gain / delivery_s, 3)
        rows.append({
            "date": L["date"],
            "pitcher_name": L["pitcher_name"],
            "pitcher_id": L["pitcher_id"],
            "p_throws": d.get("p_throws", ""),
            "base": L["base"],
            "result": L["result"],
            "run_value": L["run_value"],
            "lead_at_firstmove_ft": L["lead_at_firstmove_ft"],
            "lead_at_release_ft": L["lead_at_release_ft"],
            "gain_to_release_ft": L["gain_to_release_ft"],
            "delivery_s": d.get("delivery_s", ""),
            "release_conf": d.get("release_conf", ""),
            "usable": d.get("usable", ""),
            "analysis_method": d.get("analysis_method", ""),
            "play_id": L["play_id"],
            "avg_velocity_ftps": "" if avg_vel is None else avg_vel,
        })

    cols = ["date", "pitcher_name", "pitcher_id", "p_throws", "base", "result",
            "run_value", "lead_at_firstmove_ft", "lead_at_release_ft",
            "gain_to_release_ft", "delivery_s", "release_conf", "usable",
            "analysis_method", "play_id", "avg_velocity_ftps"]
    with open(OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader(); w.writerows(rows)

    usable_rows = [r for r in rows if r["avg_velocity_ftps"] != ""]
    vals = [float(r["avg_velocity_ftps"]) for r in usable_rows]

    print("=" * 74)
    print(f"DELIVERY-WINDOW RUNNER VELOCITY  ({RUNNER}, {YEAR})")
    print("  avg_velocity_ftps = gain_to_release_ft / delivery_s")
    print("=" * 74)
    hdr = f"{'date':10} {'pitcher':22} {'res':4} {'gain':>5} {'deliv':>6} {'vel':>7} {'meth':>9}"
    print(hdr); print("-" * len(hdr))
    for r in sorted(rows, key=lambda x: (x["avg_velocity_ftps"] == "",
                    -(float(x["avg_velocity_ftps"]) if x["avg_velocity_ftps"] != "" else 0))):
        vel = r["avg_velocity_ftps"]
        vel_s = f"{float(vel):7.2f}" if vel != "" else "    n/a"
        dv = r["delivery_s"] or "n/a"
        g = fnum(r["gain_to_release_ft"])
        g_s = f"{g:5.1f}" if g is not None else "  n/a"
        print(f"{r['date']:10} {r['pitcher_name']:22} {r['result']:4} "
              f"{g_s} {str(dv):>6} {vel_s} "
              f"{r['analysis_method']:>9}")
    print("-" * len(hdr))
    print(f"usable plays: {len(usable_rows)}/{len(rows)}")
    if vals:
        print(f"velocity ft/s  mean={st.mean(vals):.2f}  median={st.median(vals):.2f}  "
              f"min={min(vals):.2f}  max={max(vals):.2f}  std={st.pstdev(vals):.2f}")

    sb = [float(r["avg_velocity_ftps"]) for r in usable_rows if r["result"] == "SB"]
    cs = [float(r["avg_velocity_ftps"]) for r in usable_rows if r["result"] == "CS"]
    print()
    if sb:
        print(f"  SB (n={len(sb)})  mean vel = {st.mean(sb):.2f} ft/s")
    if cs:
        print(f"  CS (n={len(cs)})  mean vel = {st.mean(cs):.2f} ft/s")

    def corr(xs, ys):
        if len(xs) < 3:
            return None
        mx, my = st.mean(xs), st.mean(ys)
        num = sum((a - mx) * (b - my) for a, b in zip(xs, ys))
        dx = sum((a - mx) ** 2 for a in xs) ** 0.5
        dy = sum((b - my) ** 2 for b in ys) ** 0.5
        return num / (dx * dy) if dx and dy else None

    dlv = [float(r["delivery_s"]) for r in usable_rows]
    gns = [float(r["gain_to_release_ft"]) for r in usable_rows]
    lfm = [float(r["lead_at_firstmove_ft"]) for r in usable_rows]
    print()
    for label, c in (("delivery_s", corr(vals, dlv)),
                     ("gain_to_release_ft", corr(vals, gns)),
                     ("lead_at_firstmove", corr(vals, lfm))):
        if c is not None:
            print(f"  corr(velocity, {label:20}) = {c:+.2f}")
    print("=" * 74)
    print(f"wrote {OUT}")


if __name__ == "__main__":
    main()


In [ ]:
soto25 = ARCH / "Soto_2025" / "delivery_velocity_2025.csv"
if soto25.exists():
    df = pd.read_csv(soto25)
    print(f"Soto 2025: {len(df)} attempts")
    display(df[["date","pitcher_name","result","gain_to_release_ft",
                "delivery_s","avg_velocity_ftps"]].head(10))
else:
    print(f"Not found: {soto25}")


## Archived Runner Run: Soto 2026

### delivery_velocity — Soto 2026

In [ ]:
#!/usr/bin/env python3
"""
Delivery-window runner velocity metric (Juan Soto, 2026).

    avg_velocity_ftps = gain_to_release_ft / delivery_s

gain_to_release_ft = lead_at_release_ft - lead_at_firstmove_ft   (Statcast lead data)
delivery_s         = CV-measured first-move -> ball-release time  (cv_pilot detector)

The average ground speed (ft/s) the runner builds during the window the pitcher is
committed to the plate. Joins on play_id (the Savant GUID). Soto is a FAST runner
(contrast to Naylor) so this set probes whether a fast runner's delivery-window
velocity tracks his elite sprint speed or is still pitcher-matchup driven.

Inputs (same folder):
    soto2026_leads.csv            Statcast lead distances + outcome/run value
    pitcher_delivery_2026.csv     CV per-clip delivery time + confidence

Output:
    delivery_velocity_2026.csv    joined rows + avg_velocity_ftps
"""
import csv, os, statistics as st

HERE = os.path.dirname(os.path.abspath(__file__))
LEADS = os.path.join(HERE, "soto2026_leads.csv")
DELIV = os.path.join(HERE, "pitcher_delivery_2026.csv")
OUT = os.path.join(HERE, "delivery_velocity_2026.csv")
RUNNER = "Juan Soto"
YEAR = 2026


def load_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def fnum(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def main():
    leads = load_csv(LEADS)
    deliv = {r["play_id"]: r for r in load_csv(DELIV)}

    rows = []
    for L in leads:
        d = deliv.get(L["play_id"])
        if d is None:
            continue
        delivery_s = fnum(d.get("delivery_s"))
        usable = str(d.get("usable", "")).strip().lower() == "true"
        gain = fnum(L.get("gain_to_release_ft"))
        avg_vel = None
        if usable and delivery_s and delivery_s > 0 and gain is not None:
            avg_vel = round(gain / delivery_s, 3)
        rows.append({
            "date": L["date"],
            "pitcher_name": L["pitcher_name"],
            "pitcher_id": L["pitcher_id"],
            "p_throws": d.get("p_throws", ""),
            "base": L["base"],
            "result": L["result"],
            "run_value": L["run_value"],
            "lead_at_firstmove_ft": L["lead_at_firstmove_ft"],
            "lead_at_release_ft": L["lead_at_release_ft"],
            "gain_to_release_ft": L["gain_to_release_ft"],
            "delivery_s": d.get("delivery_s", ""),
            "release_conf": d.get("release_conf", ""),
            "usable": d.get("usable", ""),
            "analysis_method": d.get("analysis_method", ""),
            "play_id": L["play_id"],
            "avg_velocity_ftps": "" if avg_vel is None else avg_vel,
        })

    cols = ["date", "pitcher_name", "pitcher_id", "p_throws", "base", "result",
            "run_value", "lead_at_firstmove_ft", "lead_at_release_ft",
            "gain_to_release_ft", "delivery_s", "release_conf", "usable",
            "analysis_method", "play_id", "avg_velocity_ftps"]
    with open(OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader(); w.writerows(rows)

    usable_rows = [r for r in rows if r["avg_velocity_ftps"] != ""]
    vals = [float(r["avg_velocity_ftps"]) for r in usable_rows]

    print("=" * 74)
    print(f"DELIVERY-WINDOW RUNNER VELOCITY  ({RUNNER}, {YEAR})")
    print("  avg_velocity_ftps = gain_to_release_ft / delivery_s")
    print("=" * 74)
    hdr = f"{'date':10} {'pitcher':22} {'res':4} {'gain':>5} {'deliv':>6} {'vel':>7} {'meth':>9}"
    print(hdr); print("-" * len(hdr))
    for r in sorted(rows, key=lambda x: (x["avg_velocity_ftps"] == "",
                    -(float(x["avg_velocity_ftps"]) if x["avg_velocity_ftps"] != "" else 0))):
        vel = r["avg_velocity_ftps"]
        vel_s = f"{float(vel):7.2f}" if vel != "" else "    n/a"
        dv = r["delivery_s"] or "n/a"
        g = fnum(r["gain_to_release_ft"])
        g_s = f"{g:5.1f}" if g is not None else "  n/a"
        print(f"{r['date']:10} {r['pitcher_name']:22} {r['result']:4} "
              f"{g_s} {str(dv):>6} {vel_s} "
              f"{r['analysis_method']:>9}")
    print("-" * len(hdr))
    print(f"usable plays: {len(usable_rows)}/{len(rows)}")
    if vals:
        print(f"velocity ft/s  mean={st.mean(vals):.2f}  median={st.median(vals):.2f}  "
              f"min={min(vals):.2f}  max={max(vals):.2f}  std={st.pstdev(vals):.2f}")

    sb = [float(r["avg_velocity_ftps"]) for r in usable_rows if r["result"] == "SB"]
    cs = [float(r["avg_velocity_ftps"]) for r in usable_rows if r["result"] == "CS"]
    print()
    if sb:
        print(f"  SB (n={len(sb)})  mean vel = {st.mean(sb):.2f} ft/s")
    if cs:
        print(f"  CS (n={len(cs)})  mean vel = {st.mean(cs):.2f} ft/s")

    def corr(xs, ys):
        if len(xs) < 3:
            return None
        mx, my = st.mean(xs), st.mean(ys)
        num = sum((a - mx) * (b - my) for a, b in zip(xs, ys))
        dx = sum((a - mx) ** 2 for a in xs) ** 0.5
        dy = sum((b - my) ** 2 for b in ys) ** 0.5
        return num / (dx * dy) if dx and dy else None

    dlv = [float(r["delivery_s"]) for r in usable_rows]
    gns = [float(r["gain_to_release_ft"]) for r in usable_rows]
    lfm = [float(r["lead_at_firstmove_ft"]) for r in usable_rows]
    print()
    for label, c in (("delivery_s", corr(vals, dlv)),
                     ("gain_to_release_ft", corr(vals, gns)),
                     ("lead_at_firstmove", corr(vals, lfm))):
        if c is not None:
            print(f"  corr(velocity, {label:20}) = {c:+.2f}")
    print("=" * 74)
    print(f"wrote {OUT}")


if __name__ == "__main__":
    main()


### statcast_ref — Soto 2026

In [ ]:
#!/usr/bin/env python3
"""
Juan Soto 2026 Statcast base-stealing reference + head-to-head vs our CV metric.

IMPORTANT — Savant data sourcing (year correctness):
  * The basestealing-run-value LEADERBOARD CSV export IGNORES its ?year= param, so it
    CANNOT be used for a past-season reference. The PER-ATTEMPT service
        /leaderboard/services/basestealing-running-game/{rid}?season_start=Y&season_end=Y
    DOES honor the year and is what soto2026_leads.csv was built from, so the season
    SB-tracking anchors are computed from the leads CSV.
  * The sprint_speed leaderboard DOES honor ?min_season/?max_season.

Soto is a FAST runner (elite sprint speed) — the opposite profile to Naylor — so the
head-to-head asks whether his delivery-window velocity simply mirrors his top speed or
still varies by pitcher matchup.

Writes:
  statcast_ref_2026.csv         Soto's season SB metrics (one row, year-correct)
  metric_vs_statcast_2026.csv   per-play velocity joined w/ season anchors

Network: Savant has no sandbox DNS -> run with dangerouslyDisableSandbox.
"""
import csv, io, os, statistics as st, requests

HERE = os.path.dirname(os.path.abspath(__file__))
RUNNER_ID = 665742
RUNNER_NAME = "Juan Soto"
YEAR = 2026
LEADS = os.path.join(HERE, "soto2026_leads.csv")
DV = os.path.join(HERE, "delivery_velocity_2026.csv")
REF_OUT = os.path.join(HERE, "statcast_ref_2026.csv")
H2H_OUT = os.path.join(HERE, "metric_vs_statcast_2026.csv")
S = requests.Session(); S.headers.update({"User-Agent": "Mozilla/5.0"})

SPRINT = ("https://baseballsavant.mlb.com/leaderboard/sprint_speed"
          f"?attempts=1&min_season={YEAR}&max_season={YEAR}&position=&team=&csv=true")


def getcsv(u):
    return list(csv.DictReader(io.StringIO(S.get(u, timeout=30).text.lstrip("﻿"))))


def fnum(x):
    try: return float(x)
    except (TypeError, ValueError): return None


def main():
    sprint = getcsv(SPRINT)
    s = next(r for r in sprint if r.get("player_id") == str(RUNNER_ID))

    leads = list(csv.DictReader(open(LEADS)))
    sb_rows = [r for r in leads if r["result"] == "SB"]
    cs_rows = [r for r in leads if r["result"] == "CS"]
    n_sb, n_cs = len(sb_rows), len(cs_rows)
    # only rows with measured leads (some balk/FB rows have empty lead fields)
    lead_rows = [r for r in leads if fnum(r["lead_at_firstmove_ft"]) is not None]
    fm_all = [fnum(r["lead_at_firstmove_ft"]) for r in lead_rows]
    rel_all = [fnum(r["lead_at_release_ft"]) for r in lead_rows]
    gain_all = [fnum(r["gain_to_release_ft"]) for r in lead_rows]
    fm_sb = [fnum(r["lead_at_firstmove_ft"]) for r in sb_rows if fnum(r["lead_at_firstmove_ft"]) is not None]
    rel_sb = [fnum(r["lead_at_release_ft"]) for r in sb_rows if fnum(r["lead_at_release_ft"]) is not None]

    ref = {
        "player": RUNNER_NAME, "season": YEAR,
        "sprint_speed_ftps": s["sprint_speed"],
        "hp_to_1b_s": s["hp_to_1b"],
        "competitive_runs": s["competitive_runs"],
        "n_tracked_attempts": len(leads),
        "n_sb": n_sb, "n_cs": n_cs,
        "sb_success_pct": round(100 * n_sb / (n_sb + n_cs), 1) if (n_sb + n_cs) else "",
        "steal_run_value": round(sum(fnum(r["run_value"]) for r in leads if fnum(r["run_value"]) is not None), 3),
        "primary_lead_ft": round(st.mean(fm_all), 2),
        "secondary_lead_ft": round(st.mean(rel_all), 2),
        "sec_minus_prim_lead_ft": round(st.mean(gain_all), 2),
        "primary_lead_sbx_ft": round(st.mean(fm_sb), 2) if fm_sb else "",
        "secondary_lead_sbx_ft": round(st.mean(rel_sb), 2) if rel_sb else "",
        "source_note": "SB anchors from per-attempt basestealing-running-game (year-correct); "
                       "sprint from sprint_speed leaderboard; leaderboard CSV year-param is broken",
    }
    with open(REF_OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(ref)); w.writeheader(); w.writerow(ref)

    dv = list(csv.DictReader(open(DV)))
    usable = [r for r in dv if r["avg_velocity_ftps"] != ""]
    vel = [float(r["avg_velocity_ftps"]) for r in usable]
    lfm = [float(r["lead_at_firstmove_ft"]) for r in dv if r["lead_at_firstmove_ft"]]
    lrel = [float(r["lead_at_release_ft"]) for r in dv if r["lead_at_release_ft"]]
    sprint_v = fnum(ref["sprint_speed_ftps"])

    print("=" * 78)
    print(f"{RUNNER_NAME.upper()} {YEAR} — SB-TRACKING METRICS  (Statcast season + our CV metric)")
    print("=" * 78)
    print("SEASON STATCAST (year-correct; SB anchors from per-attempt service):")
    print(f"  sprint speed          {ref['sprint_speed_ftps']:>6} ft/s")
    print(f"  home-to-1B            {ref['hp_to_1b_s']:>6} s")
    print(f"  tracked attempts      {ref['n_tracked_attempts']:>6}")
    print(f"  primary lead (all)    {ref['primary_lead_ft']:>6} ft")
    print(f"  secondary lead (all)  {ref['secondary_lead_ft']:>6} ft")
    print(f"  sec-minus-prim gain   {ref['sec_minus_prim_lead_ft']:>6} ft")
    print(f"  primary lead (steals) {ref['primary_lead_sbx_ft']:>6} ft")
    print(f"  secondary lead(steals){ref['secondary_lead_sbx_ft']:>6} ft")
    print(f"  SB / CS               {ref['n_sb']} / {ref['n_cs']}  "
          f"({ref['sb_success_pct']}% success)")
    print(f"  steal run value       {ref['steal_run_value']:>+6} runs")
    print()
    if vel:
        print("OUR CV METRIC (delivery-window velocity, per-attempt — varies by matchup):")
        print(f"  mean {st.mean(vel):.2f}  median {st.median(vel):.2f}  "
              f"range {min(vel):.2f}-{max(vel):.2f} ft/s   (n={len(vel)})")
        print()
        print("-" * 78)
        print("CROSS-VALIDATION  (same per-attempt rows, sanity check):")
        if lfm:
            print(f"  mean lead@firstmove = {st.mean(lfm):5.2f} ft   (= primary lead all = {ref['primary_lead_ft']})")
        if lrel:
            print(f"  mean lead@release   = {st.mean(lrel):5.2f} ft   (= secondary lead all = {ref['secondary_lead_ft']})")
        print()
        print("-" * 78)
        print(f"HEAD-TO-HEAD — what each metric can/cannot do for these {len(usable)} attempts:")
        print(f"  sprint speed ({sprint_v:.1f} ft/s): runner-only, SAME every play -> 0 per-play resolution.")
        print(f"  our velocity (mean {st.mean(vel):.1f} ft/s): runner x PITCHER interaction ->")
        print(f"     varies {min(vel):.1f}-{max(vel):.1f} ft/s across matchups.")
        print(f"  ratio our-vel / sprint-speed = {st.mean(vel)/sprint_v:.2f}  "
              f"(~{100*st.mean(vel)/sprint_v:.0f}% of top speed during the delivery window)")
    print("=" * 78)

    rows = [(float(r["avg_velocity_ftps"]) if r["avg_velocity_ftps"] != "" else -1, r) for r in dv]
    rows.sort(key=lambda x: -x[0])
    out = []
    for v, r in rows:
        out.append({
            "date": r["date"], "pitcher_name": r["pitcher_name"], "result": r["result"],
            "avg_velocity_ftps": r["avg_velocity_ftps"],
            "lead_at_release_ft": r["lead_at_release_ft"],
            "delivery_s": r["delivery_s"], "analysis_method": r["analysis_method"],
            "soto_sprint_speed_ftps": ref["sprint_speed_ftps"],
            "soto_secondary_lead_sbx_ft": ref["secondary_lead_sbx_ft"],
        })
    with open(H2H_OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(out[0])); w.writeheader(); w.writerows(out)
    print(f"\nwrote {os.path.basename(REF_OUT)} + {os.path.basename(H2H_OUT)}")


if __name__ == "__main__":
    main()


## Pooled Multi-Runner Attempt Model

Pools all available runner-seasons, then evaluates whether adding CV delivery velocity
improves discrimination over Statcast lead features alone.


In [ ]:
# attempt_model.py — pooled LOO-CV AUC comparison across feature blocks
#!/usr/bin/env python3
"""
Pooled multi-runner per-attempt steal model — does the CV delivery-window velocity
metric add discrimination over the lead/timing features alone?

Pools every Statcast-tracked attempt we have CV deliveries for, across runners:
    Naylor   2024 + 2025 + 2026     (slow, 24.5 ft/s, high-success)
    Soto     2025 + 2026            (fast, 25.8 ft/s, 30/0)
    Vladdy / Yandy / Torres / Bichette 2025   (26-27 ft/s, LOW-success stealers)

The low-success runners are the key addition: they contribute most of the CS
negatives the pool was starved of (Soto/Naylor are near-perfect stealers), which is
exactly what the SB-success classification needs to be anything but noise. Headlines:
  - +VELOCITY−BASE classification delta (sensitive to the SB/CS balance)
  - univariate SB-vs-CS separation per feature (stable)
  - continuous run_value regression (stable)
Still treat the classification AUC as proof-of-harness — even pooled, CS are a minority.

Two targets:
  (1) y_success  = 1 if SB else 0           -> logistic, leave-one-out CV AUC
  (2) run_value  (continuous)                -> OLS, report R² + corr

Feature blocks (all PER-ATTEMPT varying — within one runner, season-constant metrics
like sprint speed cannot discriminate between his own attempts, so they are excluded
here; the cross-runner sprint-vs-velocity comparison is a separate, future test):
    BASE      = lead_at_firstmove_ft, gain_to_release_ft, lead_at_release_ft
    +DELIVERY = BASE + delivery_s
    +VELOCITY = BASE + delivery_s + avg_velocity_ftps      (the metric under test)

Headline = CV-AUC(+VELOCITY) − CV-AUC(BASE). With only ~3 CS in ~36 attempts the
classification AUC is *very* noisy (wide CIs) — this is a proof-of-harness, not a
proof-of-effect. The univariate SB-vs-CS separation table and the run_value regression
are the more stable reads at this sample size.

Output: attempt_auc.csv + printed verdict.
"""
import csv, os, sys
import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)  # cv_pilot/
SOURCES = [
    os.path.join(ROOT, "Naylor_2024", "delivery_velocity_2024.csv"),
    os.path.join(ROOT, "Naylor_2025", "delivery_velocity_2025.csv"),
    os.path.join(ROOT, "Naylor_2026", "delivery_velocity_2026.csv"),
    os.path.join(ROOT, "Soto_2025", "delivery_velocity_2025.csv"),
    os.path.join(ROOT, "Soto_2026", "delivery_velocity_2026.csv"),
    os.path.join(ROOT, "Vladdy_2025", "delivery_velocity_2025.csv"),
    os.path.join(ROOT, "Yandy_2025", "delivery_velocity_2025.csv"),
    os.path.join(ROOT, "Torres_2025", "delivery_velocity_2025.csv"),
    os.path.join(ROOT, "Bichette_2025", "delivery_velocity_2025.csv"),
    os.path.join(ROOT, "Freeman_2025", "delivery_velocity_2025.csv"),
    os.path.join(ROOT, "Gurriel_2025", "delivery_velocity_2025.csv"),
    os.path.join(ROOT, "Young_2025", "delivery_velocity_2025.csv"),
    os.path.join(ROOT, "Holliday_2025", "delivery_velocity_2025.csv"),
]

try:
    from sklearn.linear_model import LogisticRegression, LinearRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    from sklearn.model_selection import LeaveOneOut
    from sklearn.metrics import roc_auc_score
except ImportError:
    sys.exit("needs scikit-learn")

FEATS = {
    "BASE": ["lead_at_firstmove_ft", "gain_to_release_ft", "lead_at_release_ft"],
    "+DELIVERY": ["lead_at_firstmove_ft", "gain_to_release_ft", "lead_at_release_ft",
                  "delivery_s"],
    "+VELOCITY": ["lead_at_firstmove_ft", "gain_to_release_ft", "lead_at_release_ft",
                  "delivery_s", "avg_velocity_ftps"],
}


def fnum(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def load_pool():
    rows = []
    for path in SOURCES:
        if not os.path.exists(path):
            print(f"  ! missing {path} (skipping)")
            continue
        yr = os.path.basename(path).split("_")[-1][:4]
        runner = os.path.basename(os.path.dirname(path)).split("_")[0]
        for r in csv.DictReader(open(path)):
            # require a usable CV velocity (drops the few unmeasurable attempts)
            if r.get("avg_velocity_ftps", "") == "":
                continue
            rec = {
                "year": yr,
                "runner": runner,
                "date": r["date"],
                "pitcher_name": r["pitcher_name"],
                "result": r["result"],
                "y_success": 1 if r["result"] == "SB" else 0,
                "run_value": fnum(r["run_value"]),
                "lead_at_firstmove_ft": fnum(r["lead_at_firstmove_ft"]),
                "gain_to_release_ft": fnum(r["gain_to_release_ft"]),
                "lead_at_release_ft": fnum(r["lead_at_release_ft"]),
                "delivery_s": fnum(r["delivery_s"]),
                "avg_velocity_ftps": fnum(r["avg_velocity_ftps"]),
            }
            if None in (rec["delivery_s"], rec["avg_velocity_ftps"]):
                continue
            rows.append(rec)
    return rows


def loo_auc(X, y):
    """Leave-one-out CV: collect held-out predicted probs, score one global AUC."""
    y = np.asarray(y)
    if len(set(y)) < 2:
        return None
    preds = np.zeros(len(y))
    loo = LeaveOneOut()
    for tr, te in loo.split(X):
        if len(set(y[tr])) < 2:
            preds[te] = y[tr].mean()
            continue
        clf = make_pipeline(StandardScaler(),
                            LogisticRegression(max_iter=1000, C=1.0))
        clf.fit(X[tr], y[tr])
        preds[te] = clf.predict_proba(X[te])[:, 1]
    return roc_auc_score(y, preds)


def univariate_auc(rows, feat):
    """AUC of a single feature ranking SB(1) vs CS(0); flip if negatively oriented."""
    y = np.array([r["y_success"] for r in rows])
    x = np.array([r[feat] for r in rows])
    if len(set(y)) < 2:
        return None
    a = roc_auc_score(y, x)
    return max(a, 1 - a), ("+" if a >= 0.5 else "-")


def main():
    rows = load_pool()
    n = len(rows)
    n_sb = sum(r["y_success"] for r in rows)
    n_cs = n - n_sb
    print("=" * 78)
    print("POOLED MULTI-RUNNER PER-ATTEMPT STEAL MODEL — does CV velocity add over lead/timing?")
    print("=" * 78)
    by_run = {}
    for r in rows:
        by_run.setdefault(r["runner"], []).append(r)
    print(f"pooled attempts: {n}  ({n_sb} SB / {n_cs} CS)")
    print("  by runner:  " + "  ".join(
        f"{rn}:{sum(x['y_success'] for x in v)}SB/{sum(1-x['y_success'] for x in v)}CS"
        for rn, v in sorted(by_run.items())))
    print(f"** small-sample caveat: only {n_cs} CS -> classification AUC is HIGH-VARIANCE")
    print(f"   (wide CIs). Treat as proof-of-harness; univariate + run_value reads below")
    print(f"   are more stable. Real power comes when pooled with Soto et al.")
    print()

    # ---------- (1) logistic SB-success, LOO AUC for each feature block ----------
    print("-" * 78)
    print("(1) SB-success classification — leave-one-out CV AUC:")
    aucs = {}
    for name, feats in FEATS.items():
        X = np.array([[r[f] for f in feats] for r in rows])
        a = loo_auc(X, [r["y_success"] for r in rows])
        aucs[name] = a
        print(f"    {name:11} ({len(feats)} feats)  AUC = {a:.3f}" if a is not None
              else f"    {name:11}  AUC = n/a")
    if aucs.get("BASE") is not None and aucs.get("+VELOCITY") is not None:
        d = aucs["+VELOCITY"] - aucs["BASE"]
        print(f"    --> HEADLINE delta (+VELOCITY − BASE) = {d:+.3f}")

    # ---------- (2) univariate SB-vs-CS separation ----------
    print()
    print("-" * 78)
    print("(2) univariate SB-vs-CS separation (single-feature AUC, orientation):")
    uni = {}
    for f in ["avg_velocity_ftps", "delivery_s", "gain_to_release_ft",
              "lead_at_firstmove_ft", "lead_at_release_ft"]:
        res = univariate_auc(rows, f)
        if res:
            uni[f] = res[0]
            print(f"    {f:22} AUC = {res[0]:.3f}  ({res[1]} oriented)")

    # ---------- (3) run_value regression (continuous, more stable) ----------
    print()
    print("-" * 78)
    print("(3) run_value (continuous) regression — does velocity track value?:")
    yv = np.array([r["run_value"] for r in rows])
    reg_out = {}
    for name, feats in FEATS.items():
        X = np.array([[r[f] for f in feats] for r in rows])
        reg = make_pipeline(StandardScaler(), LinearRegression()).fit(X, yv)
        r2 = reg.score(X, yv)
        reg_out[name] = r2
        print(f"    {name:11} in-sample R² = {r2:.3f}")
    # univariate corr of velocity with run_value
    vv = np.array([r["avg_velocity_ftps"] for r in rows])
    c = np.corrcoef(vv, yv)[0, 1]
    print(f"    corr(avg_velocity_ftps, run_value) = {c:+.3f}")

    # ---------- means by outcome ----------
    print()
    print("-" * 78)
    print("means by outcome:")
    for lab, want in (("SB", 1), ("CS", 0)):
        sub = [r for r in rows if r["y_success"] == want]
        if sub:
            mv = np.mean([r["avg_velocity_ftps"] for r in sub])
            md = np.mean([r["delivery_s"] for r in sub])
            print(f"    {lab} (n={len(sub):2})  mean velocity = {mv:5.2f} ft/s   "
                  f"mean delivery = {md:.3f} s")

    # ---------- write ----------
    os.makedirs(HERE, exist_ok=True)
    with open(os.path.join(HERE, "attempt_auc.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["metric", "value"])
        w.writerow(["n_attempts", n]); w.writerow(["n_sb", n_sb]); w.writerow(["n_cs", n_cs])
        for k, v in aucs.items():
            w.writerow([f"loo_auc_{k}", "" if v is None else round(v, 4)])
        if aucs.get("BASE") is not None and aucs.get("+VELOCITY") is not None:
            w.writerow(["loo_auc_delta_velocity_minus_base",
                        round(aucs["+VELOCITY"] - aucs["BASE"], 4)])
        for k, v in uni.items():
            w.writerow([f"univariate_auc_{k}", round(v, 4)])
        for k, v in reg_out.items():
            w.writerow([f"runvalue_R2_{k}", round(v, 4)])
        w.writerow(["corr_velocity_runvalue", round(float(c), 4)])
    print()
    print(f"wrote {os.path.join(HERE, 'attempt_auc.csv')}")
    print("=" * 78)


if __name__ == "__main__":
    main()



In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, str(ARCH / "Naylor_model" / "attempt_model.py")],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


## Summary — What the CV Metric Adds

| Feature block | LOO-CV AUC | Δ vs BASE |
|---|---|---|
| BASE (leads only) | — | — |
| +DELIVERY | — | — |
| +VELOCITY (CV) | — | — |

Fill in from the output above. Key reads:
- **Univariate AUC** of `avg_velocity_ftps` vs SB/CS — most stable signal
- **run_value correlation** — does closing velocity track run value?
- **Classification delta** — directional, high-variance at small n

The CV metric's value is not (yet) in the AUC bump — it's in having a **per-pitch**
delivery-time estimate that replaces the league constant `LEAGUE_PITCHER_TTP = 1.30 s`
with a pitcher-specific measured value. That matters for the Naylor archetype: a slow
runner needs a slow-delivery pitcher; knowing *which* pitchers are genuinely slow (vs.
just having a weak arm) is a matchup edge.
